# Application Challenge

## Goal

Apply the geoparsing workflow to two real datasets and then design your own application idea.

## What you will do

- Inspect the Hurricane Harvey tweet dataset.
- Inspect the hydrology-related scientific paper dataset.
- Geoparse a controlled subset with the NER tools and geoparsing methods from earlier notebooks.
- Save intermediate mentions and resolved locations.
- Use the instructor demonstrations as inspiration for your own application.
- Bring or prepare your own data for a new use case.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

import json
import importlib.util
import re
import pandas as pd
import requests
from src.config import RESULTS_DIR, MAPS_DIR, GEONAMES_BASE_URL, PHOTON_BASE_URL, LLM_RAG_BASE_URL, REQUEST_TIMEOUT
from src.data_utils import save_dataframe
from src.ner_utils import extract_locations_spacy, extract_locations_stanza, extract_locations_flair, extract_locations_transformers, normalize_ner_results, combine_and_deduplicate_mentions
from src.visualization_utils import create_location_map, save_map

## Step 1: Dataset 1 - Hurricane Harvey tweets

The file `data/disaster_tweets/harvey.json` contains raw tweet text from the Hurricane Harvey crisis. Each record has a tweet id, raw text, a category, and category scores. The category was produced by an unsupervised method and can be used as weak thematic context, not as ground truth.

In [ ]:
harvey_path = PROJECT_ROOT / "data" / "disaster_tweets" / "harvey.json"
with harvey_path.open("r", encoding="utf-8") as handle:
    harvey_raw = json.load(handle)

harvey_df = pd.DataFrame([
    {
        "text_id": str(tweet_id),
        "text": item.get("text", ""),
        "category": item.get("category", ""),
        "category_scores": item.get("category_scores", {}),
    }
    for tweet_id, item in harvey_raw.items()
])

print("Harvey tweet records:", len(harvey_df))
display(harvey_df.head())
display(harvey_df["category"].value_counts().rename_axis("category").reset_index(name="count"))

## Step 2: Dataset 2 - Hydrology scientific papers

The file `data/scientific_paper/scientific_paper.jsonl` contains one paper per line. The records describe scientific publications related to hydrology and hazard events such as floods, landslides, cyclones, drought, and water resources. The source field for geoparsing is `abstract_text`. In this file, `abstract_text` starts with a repeated article title, so the notebook removes that duplicated prefix and uses the remaining abstract body as `geoparsing_text`. Fields such as publication year, title, journal, topic, DOI, and existing location metadata are kept as metadata for filtering and interpretation.

In [ ]:
paper_path = PROJECT_ROOT / "data" / "scientific_paper" / "scientific_paper.jsonl"
paper_records = []
with paper_path.open("r", encoding="utf-8") as handle:
    for line in handle:
        if line.strip():
            paper_records.append(json.loads(line))

paper_df = pd.DataFrame(paper_records)
paper_df["text_id"] = paper_df["id"].astype(str)
paper_df["abstract_text"] = paper_df["abstract_text"].fillna("").astype(str)

def abstract_body_from_source(raw_abstract, article_title):
    text = str(raw_abstract or "").strip()
    title = str(article_title or "").strip()
    if title and text.startswith(title):
        text = text[len(title):].strip()
    text = re.sub(r"^(ABSTRACT|Abstract)\s*:?", "", text).strip()
    return text

paper_df["abstract_body"] = [
    abstract_body_from_source(raw, title)
    for raw, title in zip(paper_df["abstract_text"], paper_df["article_title"])
]
paper_df["abstract_chars"] = paper_df["abstract_text"].str.len()
paper_df["abstract_body_chars"] = paper_df["abstract_body"].str.len()

print("Scientific paper records:", len(paper_df))
display(paper_df[["text_id", "pub_year", "article_title", "abstract_body", "journal", "topic", "abstract_body_chars", "locations_count"]].head())
display(paper_df["topic"].value_counts().head(12).rename_axis("topic").reset_index(name="count"))
display(paper_df["pub_year"].value_counts().sort_index().tail(15).rename_axis("pub_year").reset_index(name="count"))

## Step 3: Processing controls

Processing all records can take time, especially if you call internal services or the LLM-RAG service. Start with a small threshold such as 20 or 50 texts, then increase it after the workflow is correct. The full selected text is used for each record: tweet text for Harvey, and the cleaned abstract body derived from `abstract_text` for scientific papers.

In [ ]:
DATASET_NAME = "scientific_papers"  # options: "harvey", "scientific_papers"
MAX_TEXTS = 50

NER_TOOL = "spacy"  # options: "spacy", "stanza", "flair", "transformers"
GEOPARSING_METHOD = "unitoprank"  # options: "unitoprank", "llm_rag"

USE_GEONAMES = True  # required candidate source
USE_PHOTON = False   # False: GeoNames only; True: GeoNames + Photon

LLM_RAG_SERVICE_URL = LLM_RAG_BASE_URL  # used only when GEOPARSING_METHOD = "llm_rag"

print("Dataset:", DATASET_NAME)
print("MAX_TEXTS:", MAX_TEXTS)
print("NER_TOOL:", NER_TOOL)
print("GEOPARSING_METHOD:", GEOPARSING_METHOD)
print("Candidate sources:", "GeoNames + Photon" if USE_PHOTON else "GeoNames only")

## Step 4: Prepare a processing table

In [ ]:
def prepare_dataset(name, max_texts=50):
    if name == "harvey":
        out = harvey_df[["text_id", "text", "category"]].copy()
        out = out.rename(columns={"text": "geoparsing_text"})
        out["application_context"] = out["category"]
    elif name == "scientific_papers":
        cols = ["text_id", "abstract_body", "pub_year", "article_title", "journal", "topic", "doi"]
        out = paper_df[cols].copy()
        out = out.rename(columns={"abstract_body": "geoparsing_text"})
        out["application_context"] = out["topic"]
    else:
        raise ValueError("DATASET_NAME must be 'harvey' or 'scientific_papers'")

    out = out.dropna(subset=["geoparsing_text"]).head(max_texts).copy()
    out["geoparsing_text"] = out["geoparsing_text"].astype(str)
    return out.reset_index(drop=True)

texts = prepare_dataset(DATASET_NAME, MAX_TEXTS)
texts.head()

## Step 5: Define and preview toponym extraction

This reuses the NER tools introduced in Notebook 01. If a package or model is missing, run that tool's install cell in Notebook 01. This step defines the extraction function and previews only the first few records. The full `MAX_TEXTS` subset is processed later in Step 7.

In [ ]:
def extract_toponyms_with_selected_ner(text, ner_tool="spacy", source_text_id=None):
    if ner_tool == "spacy":
        results = extract_locations_spacy(text, model_name="en_core_web_sm")
    elif ner_tool == "stanza":
        results = extract_locations_stanza(text, lang="en")
    elif ner_tool == "flair":
        results = extract_locations_flair(text)
    elif ner_tool == "transformers":
        results = extract_locations_transformers(text)
    else:
        raise ValueError("NER_TOOL must be one of: spacy, stanza, flair, transformers")
    out = normalize_ner_results(results, source_text_id=source_text_id)
    if not out.empty and source_text_id is not None:
        out["source_text_id"] = out["source_text_id"].fillna(source_text_id)
    return out

def extract_mentions_for_texts(texts_df, ner_tool, dataset_name, save=True):
    mention_frames = []
    for i, row in texts_df.iterrows():
        if i and i % 10 == 0:
            print(f"NER processed {i}/{len(texts_df)} texts")
        mention_frames.append(
            extract_toponyms_with_selected_ner(
                row["geoparsing_text"],
                ner_tool=ner_tool,
                source_text_id=row["text_id"],
            )
        )

    out = combine_and_deduplicate_mentions(mention_frames)
    if out.empty:
        print("No toponyms were extracted. Check the selected NER tool and input texts.")
        return out
    if out["source_text_id"].isna().any():
        missing = int(out["source_text_id"].isna().sum())
        raise ValueError(f"{missing} extracted mentions are missing source_text_id.")

    out = out.merge(
        texts_df[["text_id", "geoparsing_text", "application_context"]].rename(
            columns={"text_id": "source_text_id", "geoparsing_text": "full_text"}
        ),
        on="source_text_id",
        how="left",
    )

    if save:
        mention_path = RESULTS_DIR / f"{dataset_name}_challenge_mentions.csv"
        save_dataframe(out, mention_path)
        print("Saved mentions:", mention_path)
    return out

preview_texts = texts.head(min(3, len(texts))).copy()
preview_mentions = extract_mentions_for_texts(preview_texts, NER_TOOL, DATASET_NAME, save=False)
preview_mentions.head(20)

## Step 6: Resolution helpers

Choose one geoparsing method. `unitoprank` uses the UniTopRank workflow from Notebook 03. `llm_rag` uses the LLM-RAG resolution service from Notebook 04. GeoNames is required; Photon is an optional additional candidate source.

In [ ]:
def check_candidate_sources(use_geonames=True):
    if not use_geonames:
        raise ValueError("USE_GEONAMES must stay True for these geoparsing workflows.")

def candidate_source_label(use_photon=False):
    return "geonames_photon" if use_photon else "geonames"

def resolve_with_llm_rag(texts_df, mentions_df, service_base_url, use_geonames=True, use_photon=False):
    check_candidate_sources(use_geonames)
    if not service_base_url:
        print("LLM_RAG_SERVICE_URL is not configured.")
        return pd.DataFrame()
    rows = []
    for _, text_row in texts_df.iterrows():
        group = mentions_df[mentions_df["source_text_id"] == text_row["text_id"]]
        if group.empty:
            continue
        payload = {
            "text": text_row["geoparsing_text"],
            "toponyms": [
                {"text": r["mention"], "start": int(r["start"]), "end": int(r["end"])}
                for _, r in group.iterrows()
                if pd.notna(r.get("start")) and pd.notna(r.get("end"))
            ],
            "use_geonames": True,
            "use_photon": bool(use_photon),
        }
        try:
            response = requests.post(f"{service_base_url.rstrip('/')}/resolve", json=payload, timeout=max(REQUEST_TIMEOUT, 60))
            response.raise_for_status()
            for item in response.json().get("results", []):
                rows.append({
                    "source_text_id": text_row["text_id"],
                    "mention": item.get("text") or item.get("mention"),
                    "selected_name": item.get("address") or item.get("full_name"),
                    "country": item.get("country"),
                    "lat": item.get("lat"),
                    "lon": item.get("lon"),
                    "method": "llm_rag_geonames_photon" if use_photon else "llm_rag_geonames",
                })
        except Exception as exc:
            print(f"LLM-RAG failed for {text_row['text_id']}: {exc}")
    return pd.DataFrame(rows)

def add_unitoprank_paths():
    for path in [PROJECT_ROOT / "external" / "UniTopRank", PROJECT_ROOT / "external" / "UniTopRank" / "api"]:
        if path.exists() and str(path) not in sys.path:
            sys.path.insert(0, str(path))
    if importlib.util.find_spec("unitoprank") is None and importlib.util.find_spec("unitorank") is not None:
        import importlib as importlib_module
        import unitorank
        sys.modules.setdefault("unitoprank", unitorank)
        for name in ["types", "ranker", "candidate_retriever", "ner", "pipeline"]:
            try:
                sys.modules.setdefault(f"unitoprank.{name}", importlib_module.import_module(f"unitorank.{name}"))
            except Exception:
                pass

def resolve_with_unitoprank(texts_df, mentions_df, use_geonames=True, use_photon=False):
    check_candidate_sources(use_geonames)
    candidate_source = candidate_source_label(use_photon)
    add_unitoprank_paths()
    try:
        from geo_rank_api import resolve_toponyms
        from geoparsing_api import CandidateRetriever, CandidateRetrieverConfig
    except Exception as exc:
        print("UniTopRank is not available. Run Notebook 03 setup first.", exc)
        return pd.DataFrame()

    def prefix(base_url, param):
        base = str(base_url).strip()
        if f"{param}=" in base:
            return base
        separator = "&" if "?" in base else "?"
        return f"{base}{separator}{param}="

    retriever = CandidateRetriever(CandidateRetrieverConfig(
        geonames_enabled=True,
        photon_enabled=use_photon,
        bool_merge=False,
        geonames_url=prefix(GEONAMES_BASE_URL, "location"),
        photon_url=prefix(PHOTON_BASE_URL, "q"),
        timeout_seconds=int(REQUEST_TIMEOUT),
    ))

    rows = []
    candidate_queries = 0
    total_candidates = 0
    for _, text_row in texts_df.iterrows():
        group = mentions_df[mentions_df["source_text_id"] == text_row["text_id"]]
        toponyms = [
            {"LOC": r["mention"], "start": int(r["start"]), "end": int(r["end"])}
            for _, r in group.iterrows()
            if pd.notna(r.get("start")) and pd.notna(r.get("end"))
        ]
        if not toponyms:
            continue
        candidates_by_toponym = retriever.get_candidates_for_toponyms([m["LOC"] for m in toponyms])
        candidate_queries += len(candidates_by_toponym)
        total_candidates += sum(len(candidates) for candidates in candidates_by_toponym.values())
        try:
            resolved, _ = resolve_toponyms(text=text_row["geoparsing_text"], mentions=toponyms, candidates_by_toponym=candidates_by_toponym, top_n=10)
            for item in resolved:
                selected_name = item.get("address")
                lat = item.get("lat")
                lon = item.get("lon")
                if not selected_name or lat in (None, 0, 0.0) or lon in (None, 0, 0.0):
                    continue
                rows.append({
                    "source_text_id": text_row["text_id"],
                    "mention": item.get("LOC") or item.get("text"),
                    "selected_name": selected_name,
                    "lat": lat,
                    "lon": lon,
                    "method": f"unitoprank_{candidate_source}",
                })
        except Exception as exc:
            print(f"UniTopRank failed for {text_row['text_id']}: {exc}")
    print(f"UniTopRank candidate lookup: {candidate_queries} mention queries, {total_candidates} candidates returned.")
    if candidate_queries and total_candidates == 0:
        print("No candidates were returned by GeoNames/Photon. Check DLR network/VPN and the configured candidate service URLs.")
    return pd.DataFrame(rows)

## Step 7: Run the complete geoparsing workflow

Run this cell after changing the configuration in Step 3. It prepares the selected records, extracts toponyms with the selected NER tool, resolves them with the selected geoparsing method, and saves both intermediate mentions and final resolved locations.

In [ ]:
texts = prepare_dataset(DATASET_NAME, MAX_TEXTS)
print(f"Running full geoparsing workflow on {len(texts)} {DATASET_NAME} records")

mentions = extract_mentions_for_texts(texts, NER_TOOL, DATASET_NAME)
print("Extracted mentions:", len(mentions))

if mentions.empty:
    resolved = pd.DataFrame()
elif GEOPARSING_METHOD == "unitoprank":
    resolved = resolve_with_unitoprank(
        texts,
        mentions,
        use_geonames=USE_GEONAMES,
        use_photon=USE_PHOTON,
    )
elif GEOPARSING_METHOD == "llm_rag":
    resolved = resolve_with_llm_rag(
        texts,
        mentions,
        LLM_RAG_SERVICE_URL,
        use_geonames=USE_GEONAMES,
        use_photon=USE_PHOTON,
    )
else:
    raise ValueError("GEOPARSING_METHOD must be unitoprank or llm_rag")

if resolved.empty:
    print("No resolved locations returned. Check NER output, UniTopRank setup, service access, or candidate-source connectivity.")
else:
    if "source_text_id" in resolved.columns and "application_context" not in resolved.columns:
        resolved = resolved.merge(
            texts[["text_id", "application_context"]],
            left_on="source_text_id",
            right_on="text_id",
            how="left",
        )
    resolved_path = RESULTS_DIR / f"{DATASET_NAME}_{GEOPARSING_METHOD}_challenge_results.csv"
    save_dataframe(resolved, resolved_path)
    print("Saved:", resolved_path)

resolved.head(20)

## Step 8: Summarize and visualize

In [ ]:
if not resolved.empty and {"lat", "lon"}.issubset(resolved.columns):
    summary_cols = [col for col in ["application_context", "country", "method"] if col in resolved.columns]
    if summary_cols:
        display(resolved.groupby(summary_cols, dropna=False).size().reset_index(name="count").head(30))
    else:
        display(resolved["mention"].value_counts().head(20).rename_axis("mention").reset_index(name="count"))

    fmap = create_location_map(resolved, popup_cols=[c for c in ["mention", "selected_name", "application_context", "method"] if c in resolved.columns])
    map_path = save_map(fmap, MAPS_DIR / f"{DATASET_NAME}_{GEOPARSING_METHOD}_challenge_map.html")
    print("Saved map:", map_path)
    display(fmap)
else:
    print("No valid coordinates available for mapping.")

## Instructor demonstrations

The workshop includes two demonstrations built from these raw datasets:

- A Hurricane Harvey tweet exploration demo that uses tweet text, unsupervised categories, and geoparsed locations to support crisis-situation awareness.
- A hydrology scientific-paper exploration demo that uses publication metadata, abstracts, topics, and geoparsed study locations to support location-aware literature discovery.

The demo code is not included here. Use the ideas and outputs from this notebook as inspiration for your own application.

## Bring your own data

You can use the same workflow with your own dataset. Prepare a table with `text_id` and `geoparsing_text` columns. Metadata columns such as category, year, topic, source, or author can be kept and used later for filtering or visualization.

In [ ]:
my_data = pd.DataFrame([
    {"text_id": "my_1", "geoparsing_text": "Add your own text here.", "application_context": "my use case"}
])

# Example: replace `texts` with your own table and rerun Step 5 onward.
# texts = my_data.head(MAX_TEXTS).copy()
my_data

## Exercise

Choose one dataset and process a small subset. Then design one application view or analysis question:

- Which categories or topics contain the most place references?
- Which places appear repeatedly?
- Which geoparsing method works best for your use case?
- What would an end user need to see on a map or dashboard?

In [ ]:
feedback = {
    "Dataset used": DATASET_NAME,
    "NER tool used": NER_TOOL,
    "Geoparsing method used": GEOPARSING_METHOD,
    "Candidate sources": "GeoNames + Photon" if USE_PHOTON else "GeoNames only",
    "Application idea": "",
    "What worked well": "",
    "What was difficult": "",
}
feedback

## Common issues

- Processing all records can take time; start with `MAX_TEXTS = 20` or `MAX_TEXTS = 50`.
- Optional NER packages or models may be missing; install them in Notebook 01.
- GeoNames, Photon, UniTopRank, and LLM-RAG services may require setup or DLR network/VPN access.
- Scientific papers use the full `abstract_text` field for geoparsing.